# JiT-S2-Spatial-Mamba (no gate) — global condition tokens **outside** the scan (full-m32)

`model: spatial_global` (`src/models/spatial_global.py`). **M = 32** class tokens (`y_emb` + a learned slot) join at block 4, are carried through blocks 4–11 and stripped before the final layer, as the in-context tokens. Unlike those, they are **never inserted into the scan**: image ↔ image mixing is the stock scan + SASF, and every interaction with a token is linear, **undecayed** and **order-free**, through the block's own `x_proj` / `dt_proj`.

Per block, with `v = Δ ⊙ u` for both groups (d_state 1):
- **memories:** the tokens write `S = (1/M) Σ_m v^e_m B^e_m`, the image writes `G = (1/L) Σ_s v_s B_s`, each one value per channel;
- **image:** the memory anchors the scan states **before SASF**, `h = SASF(x + α_xe ⊙ S)`, `y = C·h + D·u`. This equals the scan started from (and pulled towards) `α_xe S`, so the kernel is unchanged. With replicate padding SASF turns the constant `α_xe S` into `κ ⊙ α_xe S` (κ = Σ_j α_j Σ taps_j, 27 at init), so every cell reads it undecayed;
- **tokens:** `y^e = (α_ee ⊙ S + α_ex ⊙ G)·C^e + D·u^e`, not fused.

This run: **full, M = 32**: JiT Table 9's token count.

Differences from the in-context class tokens (`…-incontext-class3`), besides the mixing:
- the token features get the mixer DWConv's **centre tap + bias + SiLU**, where class tokens get the raw `in_proj` output;
- the gains `α_*` (384,) are ones-init with no weight decay;
- block 11 holds `α_xe` only, because its token rows are stripped right after it.

| model (Tiny-ImageNet, 100 epochs) | params | GMACs | FID-10K |
|---|---|---|---|
| Spatial, no gate, d_state 1, LPU zero | 29,686,116 | baseline | 93.95 |
| + 3 in-context class tokens (in the scan) | 29,687,268 | | ? |
| **+ global tokens, full-m32** | 29,706,852 | +0.383 (estimate) | ? |

---

JiT-S2-Spatial-Mamba (no gate) on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.

## 1. Environment  *(Internet ON)*

In [ ]:
import os, sys

# Pick the stack by the image's Python. torch 2.5.1 has no matching torchvision
# for Python 3.13 (Kaggle's newer image), so 3.13 moves to torch 2.6.0; 3.12
# keeps torch 2.5.1 exactly as every earlier run. mamba-ssm 2.2.4 and
# causal-conv1d 1.5.0.post8 publish cp312 and cp313 wheels for both.
PY = f"cp{sys.version_info.major}{sys.version_info.minor}"
if sys.version_info >= (3, 13):
    TORCH, TVISION, TAUDIO, TTAG = "2.6.0", "0.21.0", "2.6.0", "torch2.6"
else:
    TORCH, TVISION, TAUDIO, TTAG = "2.5.1", "0.20.1", "2.5.1", "torch2.5"
print("python", sys.version.split()[0], "->", PY, "| torch", TORCH)

# 1) Pin torch
!pip install -q torch=={TORCH} torchvision=={TVISION} torchaudio=={TAUDIO} \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = f"causal_conv1d-1.5.0.post8+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"
MAMBA  = f"mamba_ssm-2.2.4+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")
for w in (CAUSAL, MAMBA):
    p = f"/kaggle/working/{w}"
    assert os.path.exists(p) and os.path.getsize(p) > 0, f"wheel download failed: {w}"

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. The model (src/models/spatial_global.py), the build_model wiring
# in BOTH run_experiment.py and evaluate.py, the configs, the CPU test, the GPU
# check and the diagnostic must all be on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: confirm this config and both entry points are present
!test -f configs/tiny_imagenet/jit-s2-spatial-nogate-global-full-m32.yaml \
    && grep -q 'global_len=m.get' scripts/run_experiment.py && grep -q 'global_len=m.get' scripts/evaluate.py \
    && echo "wiring OK (config + train + eval)" || echo "WIRING MISSING -- push the spatial_global commit to main"

## 3. Verify the global tokens  *(first session only; ~3-5 min)*

In [ ]:
# === Verify spatial_global (CPU proofs, ~1 min; first session only -- comment out on resume).
# Reference scan, no GPU: global_len 0 is the JiTSpatialMamba baseline bit for bit;
# the mixer == a naive float64 transcription (anchored states + SASF, both modes,
# gate on/off); the anchored recurrence; SASF of a constant map = kappa * map;
# token permutation; S / G order-free; D counted once; bf16; counts; no dead
# parameters; guards; both build_models agree on the three configs.
# PYTHONPATH=. is REQUIRED: without it "from src.models..." fails and the
# tests' try/except misreports it as a missing mamba_ssm (known issue).
# If it FAILS, STOP -- do not train a model that is not the one reported.
!PYTHONPATH=. python tests/test_spatial_mamba_cpu.py
!PYTHONPATH=. python tests/test_spatial_global_cpu.py

In [ ]:
# === GPU check (~3-5 min; first session only). scripts/check_spatial_global_cuda.py:
# 1) the mixer on the stock CUDA scan vs mamba_ssm's selective_scan_ref (both modes,
# gate on/off, last block); 2) this config fwd+bwd under bf16: scan L = 64 in every
# block, fp32 memories, finite non-zero grads on the slots and gains; 3) it/s and
# peak memory at batch 128 vs the no-gate and in-context class3 runs; 4) fvcore
# params / GMACs vs the expected deltas -- note them for the complexity table.
# If anything FAILS, STOP.
!PYTHONPATH=. python scripts/check_spatial_global_cuda.py --config full-m32

## 4. Tiny-ImageNet  *(attached dataset, else download to /tmp)*

In [ ]:
# Tiny-ImageNet-200 -> /tmp (EPHEMERAL), ImageFolder layout for train/ and val/.
# Sources, in order: an attached Kaggle dataset containing tiny-imagenet-200/,
# then Stanford (https, http), then the SJTU mirror of the identical zip.
import os, zipfile, glob, shutil, subprocess, hashlib
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
MD5      = "90528d7ca1a48142e341f4ef8d21d0de"
SOURCES  = ["https://cs231n.stanford.edu/tiny-imagenet-200.zip",
            "http://cs231n.stanford.edu/tiny-imagenet-200.zip",
            "https://scidata.sjtu.edu.cn/records/swpcp-01e87/files/tiny-imagenet-200.zip?download=1"]

def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    attached = [os.path.dirname(p) for p in
                glob.glob("/kaggle/input/**/tiny-imagenet-200/train", recursive=True)]
    if attached:
        print("copying attached dataset:", attached[0])
        shutil.copytree(attached[0], DATA_DIR)
    else:
        if os.path.exists(ZIP) and (not zipfile.is_zipfile(ZIP) or _md5(ZIP) != MD5):
            print(f"removing broken download ({os.path.getsize(ZIP)} bytes)")
            os.remove(ZIP)
        for src in SOURCES:
            if os.path.exists(ZIP):
                break
            print("downloading", src, "(~248MB)...")
            r = subprocess.run(["wget", "-q", "--tries=3", "--timeout=60", "-O", ZIP, src])
            ok = r.returncode == 0 and zipfile.is_zipfile(ZIP) and _md5(ZIP) == MD5
            if not ok:
                size = os.path.getsize(ZIP) if os.path.exists(ZIP) else 0
                print(f"  failed (wget exit {r.returncode}, {size} bytes)")
                if os.path.exists(ZIP):
                    os.remove(ZIP)
        assert os.path.exists(ZIP), ("Tiny-ImageNet download failed from every source -- "
                                     "attach a Kaggle tiny-imagenet dataset (Add Input) and rerun this cell")
        print("extracting...")
        with zipfile.ZipFile(ZIP) as z:
            z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

n_tr, n_val = len(glob.glob(tr + "/*")), len(glob.glob(val + "/*"))
print("train classes:", n_tr, "| val classes:", n_val, "| DATA_DIR:", DATA_DIR)
assert n_tr == 200 and n_val == 200, "unexpected class count -- dataset incomplete"

## 5. Settings  *(edits the cloned config; finds the checkpoint to resume)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
GLOBAL_LEN   = 32         # M condition tokens
GLOBAL_START = 4          # block the tokens join at
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = "configs/tiny_imagenet/jit-s2-spatial-nogate-global-full-m32.yaml"
RUN     = "exp_spatial_global_full_m32"
TAG     = "spatial-global-full-m32"   # must appear in a resumed checkpoint's path (see below)
OUT_DIR = f"/kaggle/working/{RUN}"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). adaLN-Zero kept, every other spatial_mamba token / CAN option off
# (asserted by the model), so the global tokens are the only variable vs the base
# run (jit-s2-spatial-nogate.yaml).
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s, n = re.subn(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
        assert n == 1, f"key {k!r} not found in {path}"
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "global_len": GLOBAL_LEN, "global_start": GLOBAL_START,
                 "global_readback": "true", "gate": "false", "d_state": 1, "expand": 1,
                 "sasf_dilations": "[1, 2, 3]", "lpu": "zero", "ffn": "swiglu"})

# ── Resume across sessions AND accounts (automatic) ────────────────────
# Looks for THIS model's newest checkpoint in: this session's OUT_DIR, any
# attached notebook output (.../checkpoint-last.pt), and any attached DATASET
# holding an uploaded checkpoint (Kaggle unpacks an uploaded .pt into a folder
# with data.pkl / data/ / byteorder / version -- it is zipped back here).
# A checkpoint is used only if its weights match this config's model exactly
# (same names and shapes) AND its path contains TAG ("_" counts as "-"). The three
# global runs differ in shapes, but keep the tag check anyway: the tags
# (spatial-global-full-m16, spatial-global-writeonly-m16, spatial-global-full-m32)
# are not substrings of one another. Keep TAG in the name of any dataset you
# upload a checkpoint to.
# The newest epoch wins. To force one, set RESUME_CKPT_FORCE to its path.
RESUME_CKPT_FORCE = None
import os, glob, zipfile, torch, yaml
from scripts.evaluate import build_model
_ref = {k: tuple(v.shape) for k, v in build_model(yaml.safe_load(open(CONFIG))).state_dict().items()}

def _rebuild(folder):
    """Zip an unpacked checkpoint folder back into a loadable .pt in /tmp."""
    dst = os.path.join("/tmp/resume", folder.strip("/").replace("/", "__") + ".pt")
    if not os.path.exists(dst):
        os.makedirs("/tmp/resume", exist_ok=True)
        with zipfile.ZipFile(dst, "w", zipfile.ZIP_STORED) as z:
            for root, _, files in os.walk(folder):
                for f in files:
                    full = os.path.join(root, f)
                    z.write(full, os.path.join("checkpoint", os.path.relpath(full, folder)))
    return dst

cands = [os.path.join(OUT_DIR, "checkpoint-last.pt")]
cands += sorted(glob.glob("/kaggle/input/**/checkpoint-last.pt", recursive=True))
cands += [_rebuild(os.path.dirname(p))
          for p in sorted(glob.glob("/kaggle/input/**/data.pkl", recursive=True))]
best = None
print("checkpoints found:")
for c in ([RESUME_CKPT_FORCE] if RESUME_CKPT_FORCE else cands):
    if not (c and os.path.exists(c)):
        continue
    try:
        p = torch.load(c, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  unreadable ({type(e).__name__}): {c}")
        continue
    shapes = {k: tuple(v.shape) for k, v in p.get("model", {}).items()} == _ref
    match = shapes and (c == RESUME_CKPT_FORCE or TAG in c.lower().replace("_", "-"))
    label = "THIS model " if match else ("other arm? " if shapes else "other model")
    print(f"  {label}  epoch {p.get('epoch')}  {c}")
    if match and (best is None or p["epoch"] > best[0]):
        best = (p["epoch"], c)
    del p
RESUME_CKPT = best[1] if best else None

print("config :", CONFIG, "| global tokens:", GLOBAL_LEN, "from block", GLOBAL_START, "| readback true | spatial_global no gate, d_state 1, LPU zero, adaLN-Zero")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR)
print("resume :", f"{RESUME_CKPT} (last finished epoch {best[0]})" if best else "(fresh start)")

## 6. Train  *(Save Version before 12h; next session it resumes by itself)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + the model knobs now live in the (edited) config; we only override run-specific
# knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}

## 7. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
import glob, os

# Find the checkpoint to evaluate: this session's OUT_DIR first, then any
# attached notebook output (Save Version -> attach it as an input). Searches
# instead of hardcoding a notebook hash, so nothing needs editing after the
# first Save Version.
cands = []
for root in (os.path.dirname(OUT_DIR), "/kaggle/input"):
    for which in ("checkpoint-best.pt", "checkpoint-last.pt"):
        cands += sorted(glob.glob(f"{root}/**/{RUN}/{which}", recursive=True))
assert cands, f"no checkpoint found for {RUN} -- train first, or attach the saved Version"
EVAL_CKPT = cands[0]
EVAL_OUT  = f"/kaggle/working/{RUN}/eval"
print("eval ckpt:", EVAL_CKPT)

EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--n_samples 10000 --batch_size 200 --ema 1 "
            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0 --out_dir {EVAL_OUT}")
print(EVAL_CMD, "\n" + "="*70)
!PYTHONPATH=. {EVAL_CMD}

## 8. Diagnostic  *(token memories; needs the eval checkpoint and DATA_DIR/train)*

In [ ]:
# scripts/measure_spatial_global.py: per block, over t and class,
#   r_xe       = ||C SASF(alpha_xe S)|| / ||C SASF(states)||  (how much the image reads the tokens)
#   g_over_s   = ||G|| / ||S||  (NaN in write-only and in the last block)
#   class_frac / time_frac of the mean memory S(t, y)
# 4 training images per class (the only labelled split), 11 values of t, EMA1.
DIAG_OUT = f"/kaggle/working/{RUN}/global_diagnostic"
DIAG_CMD = (f"python scripts/measure_spatial_global.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--ema 1 --n-per-class 4 --t-steps 11 --out-dir {DIAG_OUT}")
print(DIAG_CMD, "\n" + "="*70)
!PYTHONPATH=. {DIAG_CMD}

## Notes
- Checkpoints (`-last`/`-best`/`-ep###`) with both EMA copies + optimizer are in `OUT_DIR` under `/kaggle/working`, so a resumed run equals a continuous one.
- **Resuming, also on another Kaggle account:** attach the previous session's notebook output, or upload its `checkpoint-last.pt` as a dataset **with `spatial-global-full-m32` in the dataset name** and attach that.
  - The settings cell keeps only checkpoints with this model's shapes and that tag in the path, and resumes from the newest epoch.
  - A baseline, in-context or other global checkpoint shows as `other model` (different shapes).
  - To use a checkpoint regardless, set `RESUME_CKPT_FORCE`. Check the `resume :` line before training.
- **Data:** the Tiny-ImageNet cell uses an attached dataset first, then Stanford, then the SJTU mirror of the identical zip (MD5-checked). To skip downloads for good, upload `tiny-imagenet-200.zip` once as your own Kaggle dataset and attach it.
- Keep `EPOCHS`, `cfg_scale`, `DATA_DIR`, seed and sampler identical to the base run (93.95) and the class3 run, so the only variable is how the tokens are connected.
- The scan stays at L = 64 in every block. Record the it/s and peak memory printed by the GPU check for the complexity table.
- **Comparisons:**
  - vs the no-gate base: what the global tokens add.
  - vs in-context class3: tokens in the scan vs outside it.
  - full vs write-only: what reading the image (G) adds.
  - M = 16 vs M = 32: the token count.
- `spatial_global` keys: `global_len`, `global_start`, `global_readback`. Every other spatial_mamba token / CAN option (`in_context_*`, `aux_tokens`, `dwconv_can`, `sasf_can`, `ssm_norm`, `feature_rope`, d_state ≠ 1) is asserted off. To try another start block, change `GLOBAL_START` and `RUN` together, keeping `spatial-global-full-m32` in the path of a resumed checkpoint.